# Finetuning different types of Transformer Models.

## 01. Causal Language Modeling (CLM)
Used by <b>decoder</b> models like GPT, this approach predicts the next token based on all previous tokens in the sequence. The model can only use context from the left (previous tokens) to predict the next token.

#### Load ELI5 Dataset
Start by loading the first 5000 examples from the <u>ELI5-Category</u> dataset with the 🤗 Datasets library. This’ll give you a chance to experiment and make sure everything works before spending more time training on the full dataset.

In [3]:
from datasets import load_dataset

eli5 = load_dataset("eli5_category", split="train[:5000]", trust_remote_code=True)

# Split the dataset
eli5 = eli5.train_test_split(test_size=0.2)

# Checking one example
eli5["train"][0]

{'q_id': '5n67qf',
 'title': 'Digitally signing message.',
 'selftext': '',
 'category': 'Technology',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dc979ck'],
  'text': ['Public Key Cryptography in a nutshell: You have two keys: a public key (that you can give out to anybody) and a private key (that — you guessed it — you keep private). Both of those keys can be used to either encrypt or decrypt messages. If you encrypt with one key, you can decrypt with the other. By giving out your public key, people can encrypt messages with it so that they know only you can read them (by decrypting with the private key). Encrypting with the private key is useful for digital signatures — we\'ll get there in a minute. Hash functions in a nutshell: Cryptographic Hash Functions are mathematical constructs that allow you to summarise data into a fixed size summary (called a "digest"), in a way that for the same message you always get the same digest, where you can\'t recover the original me

#### Preprocess

In [4]:
# Load DistilGPT2
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilgpt2"
)

You’ll notice from the example above, the <u>text</u> field is actually nested inside <u>answers</u>. This means you’ll need to extract the <u>text</u> subfield from its nested structure with the [flatten](https://huggingface.co/docs/datasets/process#flatten) method:

In [5]:
eli5 = eli5.flatten()

eli5["train"][0]

{'q_id': '5n67qf',
 'title': 'Digitally signing message.',
 'selftext': '',
 'category': 'Technology',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dc979ck'],
 'answers.text': ['Public Key Cryptography in a nutshell: You have two keys: a public key (that you can give out to anybody) and a private key (that — you guessed it — you keep private). Both of those keys can be used to either encrypt or decrypt messages. If you encrypt with one key, you can decrypt with the other. By giving out your public key, people can encrypt messages with it so that they know only you can read them (by decrypting with the private key). Encrypting with the private key is useful for digital signatures — we\'ll get there in a minute. Hash functions in a nutshell: Cryptographic Hash Functions are mathematical constructs that allow you to summarise data into a fixed size summary (called a "digest"), in a way that for the same message you always get the same digest, where you can\'t recover the original

Each subfield is now a separate column as indicated by the <u>answers</u> prefix, and the <u>text</u> field is a list now. Instead of tokenizing each sentence separately, convert the list to a string so you can jointly tokenize them.

Here is a first preprocessing function to join the list of strings for each example and tokenize the result:

In [6]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

To apply this preprocessing function over the entire dataset, use the 🤗 Datasets [map](https://huggingface.co/docs/datasets/v4.0.0/en/package_reference/main_classes#datasets.Dataset.map) method. You can speed up the <u>map</u> function by setting <u>batched=True</u> to process multiple elements of the dataset at once, and increasing the number of processes with <u>num_proc</u>. Remove any columns you don’t need:

In [7]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1774 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2167 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (5260 > 1024). Running this sequence through the model will result in indexing errors
Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1574 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1197 > 1024). Running this sequence

This dataset contains the token sequences, but some of these are longer than the maximum input length for the model.

You can now use a second preprocessing function to
- concatenate all the sequences
- split the concatenated sequences into shorter chunks defined by <u>block_size</u>, which should be both shorter than the maximum input length and short enough for your GPU RAM.

In [8]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# Apply the group_texts function over the entire dataset:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4): 100%|██████████| 1000/1000 [00:00<00:00, 4579.54 examples/s]


Now create a batch of examples using [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling). It’s more efficient to dynamically pad the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

In [9]:
# Use the end-of-sequence token as the padding token and set mlm=False.
# This will use the inputs as labels shifted to the right by one element:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#### Train

You’re ready to start training your model now! Load DistilGPT2 with [AutoModelForCausalLM](https://huggingface.co/docs/transformers/v4.53.3/en/model_doc/auto#transformers.AutoModelForCausalLM):

In [10]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

At this point, only three steps remain:

1. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.TrainingArguments). The only required parameter is output_dir which specifies where to save your model. You’ll push this model to the Hub by setting push_to_hub=True (you need to be signed in to Hugging Face to upload your model).

2. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer) along with the model, datasets, and data collator.

3. Call [train()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

In [11]:
training_args = TrainingArguments(
    output_dir="koh_eli5_clm-model",
    eval_strategy="epoch",
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

/tmp/ipykernel_19993/1787368983.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.923700,3.833283
2,3.814300,3.821710
3,3.753600,3.818210
4,3.710900,3.819015
5,3.666700,3.820701
6,3.645500,3.824158
7,3.617100,3.824773
8,3.597800,3.830066
9,3.593600,3.832611
10,3.586300,3.832649


  2025-08-21T21:58:35.551703Z  WARN  Status Code: 502. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:220



TrainOutput(global_step=13030, training_loss=3.6919204331322626, metrics={'train_runtime': 3244.7855, 'train_samples_per_second': 32.116, 'train_steps_per_second': 4.016, 'total_flos': 3403716797399040.0, 'train_loss': 3.6919204331322626, 'epoch': 10.0})

Once training is completed, use the [evaluate()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.evaluate) method to evaluate your model and get its perplexity:

In [12]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 46.18


Then share your model to the Hub with the [push_to_hub()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/trainer#transformers.Trainer.push_to_hub) method so everyone can use your model:

In [13]:
trainer.push_to_hub()

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                :   0%|          | 5.71kB /  328MB,   ???B/s  






Processing Files (1 / 2)                :   0%|          |  555kB /  328MB,  916kB/s  


Processing Files (1 / 2)                :   1%|          | 2.75MB /  328MB, 3.43MB/s  


Processing Files (1 / 2)                :   2%|▏         | 4.95MB /  328MB, 4.94MB/s  


Processing Files (1 / 2)                :   4%|▎         | 11.9MB /  328MB, 9.89MB/s  


Processing Files (1 / 2)                :   5%|▌         | 16.8MB /  328MB, 12.0MB/s  


Processing Files (1 / 2)                :   6%|▋         | 20.7MB /  328MB, 12.9MB/s  


Processing Files (1 / 2)                :   7%|▋         | 23.4MB /  328MB, 13.0MB/s  


Processing Files (1 / 2)                :   8%|▊         | 25.1MB /  328MB, 12.5MB/s  


Processing Files (1 / 2)                :   9%|▉         | 30.6MB /  328MB, 13.9MB/s  


Processing Files (1 / 

CommitInfo(commit_url='https://huggingface.co/koh43/koh_eli5_clm-model/commit/b16e09d3f389b52fe6ad3ab0f5cf0b4da98401ed', commit_message='End of training', commit_description='', oid='b16e09d3f389b52fe6ad3ab0f5cf0b4da98401ed', pr_url=None, repo_url=RepoUrl('https://huggingface.co/koh43/koh_eli5_clm-model', endpoint='https://huggingface.co', repo_type='model', repo_id='koh43/koh_eli5_clm-model'), pr_revision=None, pr_num=None)

#### Inference

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/pipelines#transformers.pipeline). Instantiate a pipeline for text generation with your model, and pass your text to it:

In [14]:
# Prompt you’d like to generate text from...
prompt = "Somatic hypermutation allows the immune system to"

from transformers import pipeline

generator = pipeline("text-generation", model="koh43/koh_eli5_clm-model")
generator(prompt)

Device set to use cuda:0


[{'generated_text': "Somatic hypermutation allows the immune system to suppress the growth of cancerous cells. The immune system can't suppress the growth of cancerous cells, but it can suppress the growth and proliferation of cancerous cells. And it's a lot more efficient. Because of how many cells it has, the immune system only has about 10% of the available cells. If you look at how many cells it requires to be harvested from - and how many times it takes to destroy - you'll see that there is ~10^10 of available cells in the body that need to be destroyed. So in order to make things even more complicated, we are going to need to look at how many cells are needed to produce cancerous cells, and how much power it can produce at a given time. What we're looking at is how many of the cells we need to produce cancerous cells. We're looking at how much power it can have at the time, and how much power it can have at the time. So we're going to look at how many people need to be destroyed.

In [15]:
from transformers import AutoTokenizer

# Tokenize the text and return the input_ids as PyTorch tensors:
tokenizer = AutoTokenizer.from_pretrained("koh43/koh_eli5_clm-model")
inputs = tokenizer(prompt, return_tensors="pt").input_ids

Use the [generate()](https://huggingface.co/docs/transformers/v4.53.3/en/main_classes/text_generation#transformers.GenerationMixin.generate) method to generate text. For more details about the different text generation strategies and parameters for controlling generation, check out the [Text generation strategies](https://huggingface.co/docs/transformers/generation_strategies) page.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("koh43/koh_eli5_clm-model")
outputs = model.generate(
    inputs,
    max_new_tokens=100,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

# Decode the generated token ids back into text:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


["Somatic hypermutation allows the immune system to shut down a new infection so they don't get sick. The brain can stop these early antibodies and prevent it from spread to other cells. When you're in remission, they can take action and keep you out of life. It's actually a lot harder on you in the first place. If you have the brain and it works, it doesn't work very well. But if you're not really immune, you can stop them from spread. This is when your immune system stops doing the usual stuff"]